In [1]:
import os
import gzip
import json
import pandas as pd
import random

# PATHS
RAW_DATA_DIR = r"C:\Users\HP\Desktop\thesis_preprocessing\data\raw"
PROCESSED_DIR = r"C:\Users\HP\Desktop\thesis_preprocessing\data\processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

LANGUAGES = ["go", "java", "javascript", "php", "python", "ruby"]
BOILERPLATE = ["copyright", "license", "author", "all rights reserved", "auto-generated"]

print("Environment Ready.")

Environment Ready.


In [2]:
def process_language_robust(language, sample_size=2000):
    print(f"--- Processing: {language.upper()} ---")
    search_path = os.path.join(RAW_DATA_DIR, language, "final", "jsonl")
    
    clean_records = []
    
    for split in ["train", "valid", "test"]:
        split_dir = os.path.join(search_path, split)
        if not os.path.exists(split_dir): continue
            
        files = [f for f in os.listdir(split_dir) if f.endswith(".jsonl.gz")]
        
        for file in files:
            file_path = os.path.join(split_dir, file)
            # Use explicit encoding to prevent Windows character errors
            with gzip.open(file_path, 'rt', encoding='utf-8', errors='ignore') as f:
                for line in f:
                    try:
                        data = json.loads(line)
                        code = data.get("code", "").strip()
                        docstring = data.get("docstring", "").strip()
                        
                        # GUARDRAIL 1: Length (BERT limit)
                        if len(data.get("code_tokens", [])) > 512:
                            continue
                        
                        # GUARDRAIL 2: Boilerplate and Empty
                        if not docstring or any(bp in docstring.lower() for bp in BOILERPLATE):
                            continue
                            
                        # GUARDRAIL 3: Multi-line Check (Prevents fragments)
                        # An actual function usually has more than 2 lines.
                        if len(code.splitlines()) < 2:
                            continue

                        clean_records.append({
                            "language": language,
                            "original_code": code, # Full string with \n preserved
                            "original_comment": docstring,
                            "token_count": len(data.get("code_tokens", []))
                        })
                    except:
                        continue
                        
    print(f"Total valid multi-line functions found: {len(clean_records)}")
    
    # Stratified Sampling
    if len(clean_records) >= sample_size:
        sampled = random.sample(clean_records, sample_size)
    else:
        print(f"Warning: Only found {len(clean_records)} for {language}")
        sampled = clean_records
        
    return pd.DataFrame(sampled)

In [3]:
all_dfs = []
for lang in LANGUAGES:
    df_lang = process_language_robust(lang)
    all_dfs.append(df_lang)

master_df = pd.concat(all_dfs, ignore_index=True)

# Save as V2 to avoid confusion
output_path = os.path.join(PROCESSED_DIR, "01_MASTER_Original_12k_V2.csv")
# CRITICAL: We use escapechar to ensure newlines are not corrupted during save
master_df.to_csv(output_path, index=False, quoting=1) # quoting=1 means quote all strings

print(f"\nSUCCESS: 12,000 samples saved to {output_path}")

--- Processing: GO ---
Total valid multi-line functions found: 336833
--- Processing: JAVA ---
Total valid multi-line functions found: 482408
--- Processing: JAVASCRIPT ---
Total valid multi-line functions found: 131763
--- Processing: PHP ---
Total valid multi-line functions found: 555541
--- Processing: PYTHON ---
Total valid multi-line functions found: 443500
--- Processing: RUBY ---
Total valid multi-line functions found: 52238

SUCCESS: 12,000 samples saved to C:\Users\HP\Desktop\thesis_preprocessing\data\processed\01_MASTER_Original_12k_V2.csv


In [4]:
print("--- DATA INTEGRITY VALIDATION ---")
# Calculate average lines of code per language
master_df['line_count'] = master_df['original_code'].apply(lambda x: len(str(x).splitlines()))

stats = master_df.groupby('language')['line_count'].agg(['mean', 'median', 'min', 'max'])
print(stats)

# Final check for PHP
php_lines = stats.loc['php', 'median']
if php_lines <= 1:
    print("\nALERT: DATA STILL CORRUPTED. PHP median line count is 1.")
else:
    print("\nVALIDATION PASSED: Data contains multi-line functions.")

--- DATA INTEGRITY VALIDATION ---
               mean  median  min  max
language                             
go          13.6905     8.0    3  122
java        14.7615    10.0    3  119
javascript  21.3020    15.0    3  302
php         17.2465    13.0    3  147
python      24.8335    18.0    3  230
ruby        12.7080     9.0    3  226

VALIDATION PASSED: Data contains multi-line functions.
